# 05 — Train the coverage model

Learns the relationship between *where a branch is* and *how much of a tract it
reaches*, then uses it to score a location that has no library yet.

| | |
|---|---|
| **Reads** | `results/coverage_details.parquet`, `results/coverage_scores.parquet`, `clean/branches.csv`, `clean/population_demographics.csv` |
| **Writes** | `models/coverage-model/model.json`, `results/coverage_predictions.parquet`, `results/coverage_predictions.csv` |
| **Needs** | `cpl-s3`; MLflow optional |

**MLflow works both ways.** Interactively, the workbench annotation gets you
`MLFLOW_TRACKING_URI`. As a pipeline node, the DSPA's
`mlflow.injectUserEnvVars: true` hands the task its own nested run. Nothing to
configure per node either way. If neither applies the notebook skips logging
rather than failing.


### Running this as an Elyra node

Drag this notebook onto a pipeline canvas, then open its **node properties**:

| Property | Value |
|---|---|
| Runtime image | Datascience with Python 3.12 (or any image with pip access) |
| Kubernetes secret | `cpl-s3` → `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_S3_ENDPOINT`, `AWS_S3_BUCKET` |
| Environment variables | `MODE`, `TRAVEL_TIME`, `NEW_BRANCH_LAT`, `NEW_BRANCH_LON` (all optional) |

The notebook installs its own dependencies in the first cell, so any runtime
image works. Everything it reads and writes lives in S3 — nothing is passed
between nodes as a file, so node ordering is the only dependency.

In [ ]:
%pip install -q boto3 pandas pyarrow numpy scikit-learn xgboost shapely pyproj mlflow

In [ ]:
import io
import os
import logging

import boto3
import pandas as pd

logging.basicConfig(level=logging.INFO, force=True)
log = logging.getLogger("coverage_model")

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    verify=False,
)
BUCKET = os.environ["AWS_S3_BUCKET"]
log.info("Bucket: %s", BUCKET)

## Parameters

The default coordinates are downtown Chicago. Point them somewhere underserved
and rerun the last two cells to see the prediction change.

In [ ]:
import math

import numpy as np
from pyproj import Geod
from shapely import wkt

geod = Geod(ellps="WGS84")

MODE = os.environ.get("MODE", "walking")
TRAVEL_TIME = int(os.environ.get("TRAVEL_TIME", "10"))
NEW_BRANCH_LAT = float(os.environ.get("NEW_BRANCH_LAT", "41.8781"))
NEW_BRANCH_LON = float(os.environ.get("NEW_BRANCH_LON", "-87.6298"))

# The furthest real coverage in the data is about 6.5 km. Branch/tract pairs
# beyond this are trivially zero and would swamp the training set with easy
# negatives -- the model would learn "predict 0" and score well doing it.
MAX_NEGATIVE_KM = 8.0

log.info("Predicting for (%s, %s) via %s/%dmin",
         NEW_BRANCH_LAT, NEW_BRANCH_LON, MODE, TRAVEL_TIME)

## Geometry helpers

The model never sees a polygon. Each branch/tract pair is reduced to *how far
apart* and *in what direction* — bearing is split into sine and cosine so that
359° and 1° come out as neighbours instead of opposites.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2
         + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2) ** 2)
    return r * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def bearing_deg(lat1, lon1, lat2, lon2):
    dlon = math.radians(lon2 - lon1)
    lat1, lat2 = math.radians(lat1), math.radians(lat2)
    x = math.sin(dlon) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(dlon)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

## Load

In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key="results/coverage_details.parquet")
coverage = pd.read_parquet(io.BytesIO(obj["Body"].read()))
log.info("Loaded %d coverage detail records", len(coverage))

obj = s3.get_object(Bucket=BUCKET, Key="clean/branches.csv")
branches = pd.read_csv(io.BytesIO(obj["Body"].read()))
branches = branches[branches["PERMANENTLY_CLOSED"] == 0].reset_index(drop=True)
branch_coords = branches.set_index("BRANCH")[["LATITUDE", "LONGITUDE"]].to_dict("index")
log.info("Loaded %d active branches", len(branches))

obj = s3.get_object(Bucket=BUCKET, Key="clean/population_demographics.csv")
tracts = pd.read_csv(io.BytesIO(obj["Body"].read()))

tract_info = {}
for _, row in tracts.iterrows():
    geom = wkt.loads(row["geometry"])
    tract_info[row["geoid"]] = {
        "centroid_lat": geom.centroid.y,
        "centroid_lon": geom.centroid.x,
        "area_sqkm": abs(geod.geometry_area_perimeter(geom)[0]) / 1e6,
        "total_population": row.get("total population", 0),
    }
log.info("Loaded %d census tracts", len(tract_info))

## Build the training set

Positives come straight from step 03. Negatives have to be invented — a tract a
branch does *not* reach produces no row, but the model needs to see those to
learn where coverage stops. `MAX_NEGATIVE_KM` keeps that from becoming the
entire dataset.

In [ ]:
modes_tt = coverage[["mode", "travel_time"]].drop_duplicates().values.tolist()
all_geoids = set(tracts["geoid"])
all_branches = set(coverage["branch"].unique())

train_rows = []

for m, tt in modes_tt:
    subset = coverage[(coverage["mode"] == m) & (coverage["travel_time"] == tt)]
    covered_pairs = set(zip(subset["branch"], subset["geoid"]))

    def row_for(bc, ti, overlap):
        dist = haversine_km(bc["LATITUDE"], bc["LONGITUDE"], ti["centroid_lat"], ti["centroid_lon"])
        bear = bearing_deg(bc["LATITUDE"], bc["LONGITUDE"], ti["centroid_lat"], ti["centroid_lon"])
        return {
            "distance_km": dist,
            "bearing_sin": math.sin(math.radians(bear)),
            "bearing_cos": math.cos(math.radians(bear)),
            "tract_area_sqkm": ti["area_sqkm"],
            "travel_time": tt,
            "mode_driving": 1 if m == "driving" else 0,
            "mode_cycling": 1 if m == "cycling" else 0,
            "mode_walking": 1 if m == "walking" else 0,
            "overlap": overlap,
        }

    for _, row in subset.iterrows():
        if row["branch"] in branch_coords and row["geoid"] in tract_info:
            train_rows.append(row_for(branch_coords[row["branch"]],
                                      tract_info[row["geoid"]],
                                      row["overlap_proportion"]))

    for b in all_branches:
        if b not in branch_coords:
            continue
        bc = branch_coords[b]
        for g in all_geoids:
            if (b, g) in covered_pairs or g not in tract_info:
                continue
            ti = tract_info[g]
            if haversine_km(bc["LATITUDE"], bc["LONGITUDE"],
                            ti["centroid_lat"], ti["centroid_lon"]) > MAX_NEGATIVE_KM:
                continue
            train_rows.append(row_for(bc, ti, 0.0))

train_df = pd.DataFrame(train_rows)
log.info("Training data: %d rows (%d positive, %d negative)",
         len(train_df), (train_df["overlap"] > 0).sum(), (train_df["overlap"] == 0).sum())
train_df.head()

## Train

In [ ]:
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

feature_cols = ["distance_km", "bearing_sin", "bearing_cos", "tract_area_sqkm",
                "travel_time", "mode_driving", "mode_cycling", "mode_walking"]

# float64 throughout: MLflow records the dtype of every column in the model
# signature and rejects an int column that arrives as a float at serving time.
X = train_df[feature_cols].astype("float64")
y = train_df["overlap"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    objective="reg:squarederror",
    random_state=42,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
log.info("RMSE: %.4f, R2: %.4f", rmse, r2)

pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)

## Log to MLflow

This cell behaves differently depending on how the notebook is being run, and
the difference matters:

- **Interactively in this workbench** — `MLFLOW_TRACKING_URI` is injected
  because the workbench carries the `opendatahub.io/mlflow-instance`
  annotation. The cell opens its own run under the `cpl-coverage-model`
  experiment.
- **As a pipeline node** — the pipeline server has already opened a parent run
  for the pipeline and a nested run for this task, and passed the task's id in
  `MLFLOW_RUN_ID`. The cell logs into *that* run instead of starting a new one,
  so the metrics stay attached to the pipeline run that produced them. This
  only works because the DSPA sets `mlflow.injectUserEnvVars: true`.
- **Neither** — logging is skipped and the model is still written to S3.

In [ ]:
tracking_uri = os.environ.get("MLFLOW_TRACKING_URI")
injected_run_id = os.environ.get("MLFLOW_RUN_ID")

if not tracking_uri:
    log.info("MLFLOW_TRACKING_URI not set, skipping MLflow logging")
else:
    import mlflow
    from mlflow.models.signature import infer_signature

    log.info("MLflow at %s (run_id from platform: %s)", tracking_uri, injected_run_id or "none")

    if injected_run_id:
        # Attach to the nested run the pipeline server already opened for this
        # task. Calling set_experiment here would fight MLFLOW_EXPERIMENT_ID.
        run_ctx = mlflow.start_run(run_id=injected_run_id)
    else:
        mlflow.set_experiment("cpl-coverage-model")
        run_ctx = mlflow.start_run(run_name=f"lab-{MODE}-{TRAVEL_TIME}min")

    with run_ctx as run:
        mlflow.log_params({
            "n_estimators": 200, "max_depth": 6, "learning_rate": 0.1,
            "mode": MODE, "travel_time": TRAVEL_TIME,
            "train_rows": len(X_train), "test_rows": len(X_test),
            "positive_samples": int((train_df["overlap"] > 0).sum()),
            "negative_samples": int((train_df["overlap"] == 0).sum()),
            "new_branch_lat": NEW_BRANCH_LAT, "new_branch_lon": NEW_BRANCH_LON,
        })
        mlflow.log_metrics({"rmse": rmse, "r2": r2})
        for feat, imp in zip(feature_cols, model.feature_importances_.tolist()):
            mlflow.log_metric(f"importance_{feat}", imp)

        model_info = mlflow.xgboost.log_model(
            model,
            name="coverage_model",
            signature=infer_signature(X_test, y_pred),
            registered_model_name="cpl-coverage-model",
        )
        log.info("MLflow run %s -> %s", run.info.run_id, model_info.model_uri)

## Save a serving copy

KServe downloads an entire prefix and MLServer wants a native xgboost file, so
the model gets its own directory. A bare key alongside the other objects would
make KServe pull those too.

In [ ]:
model.save_model("/tmp/model.json")
with open("/tmp/model.json", "rb") as fh:
    s3.put_object(Bucket=BUCKET, Key="models/coverage-model/model.json", Body=fh.read())
log.info("Wrote s3://%s/models/coverage-model/model.json", BUCKET)

## Predict for the candidate location

Score every tract as if a branch existed at the candidate coordinates, then
compare against what it already has.

In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key="results/coverage_scores.parquet")
current_scores = pd.read_parquet(io.BytesIO(obj["Body"].read()))
current_scores = current_scores[(current_scores["mode"] == MODE)
                                & (current_scores["travel_time"] == TRAVEL_TIME)]
current_map = current_scores.set_index("geoid")["percent_covered"].to_dict()

pred_rows = []
for geoid, ti in tract_info.items():
    bear = bearing_deg(NEW_BRANCH_LAT, NEW_BRANCH_LON, ti["centroid_lat"], ti["centroid_lon"])
    pred_rows.append({
        "geoid": geoid,
        "distance_km": haversine_km(NEW_BRANCH_LAT, NEW_BRANCH_LON,
                                    ti["centroid_lat"], ti["centroid_lon"]),
        "bearing_sin": math.sin(math.radians(bear)),
        "bearing_cos": math.cos(math.radians(bear)),
        "tract_area_sqkm": ti["area_sqkm"],
        "travel_time": TRAVEL_TIME,
        "mode_driving": 1 if MODE == "driving" else 0,
        "mode_cycling": 1 if MODE == "cycling" else 0,
        "mode_walking": 1 if MODE == "walking" else 0,
        "total_population": ti["total_population"],
    })

pred_df = pd.DataFrame(pred_rows)
pred_df["predicted_overlap"] = model.predict(pred_df[feature_cols].astype("float64")).clip(0, 1)
pred_df["current_coverage"] = pred_df["geoid"].map(current_map).fillna(0)
pred_df["new_coverage"] = (pred_df["current_coverage"] + pred_df["predicted_overlap"]).clip(0, 1)
pred_df["delta"] = pred_df["new_coverage"] - pred_df["current_coverage"]

result = pred_df[["geoid", "total_population", "current_coverage", "predicted_overlap",
                  "new_coverage", "delta", "distance_km"]].copy()
result = result.sort_values("delta", ascending=False).reset_index(drop=True)
result.head(10)

## Write predictions

In [ ]:
buf = io.BytesIO()
result.to_parquet(buf, index=False)
buf.seek(0)
s3.put_object(Bucket=BUCKET, Key="results/coverage_predictions.parquet", Body=buf.getvalue())

buf = io.BytesIO()
result.to_csv(buf, index=False)
buf.seek(0)
s3.put_object(Bucket=BUCKET, Key="results/coverage_predictions.csv", Body=buf.getvalue())

affected = result[result["delta"] > 0.01]
log.info(
    "New branch at (%s, %s) [%s/%dmin]: %d tracts gain >1%% coverage, "
    "%s people affected, max delta %.3f",
    NEW_BRANCH_LAT, NEW_BRANCH_LON, MODE, TRAVEL_TIME,
    len(affected), f"{affected['total_population'].astype(float).sum():,.0f}",
    result["delta"].max(),
)